In [17]:
import json
from fuzzywuzzy import process
import nltk
from nltk.corpus import stopwords

# Path to MITRE ATT&CK techniques JSON
MITRE_DICT_PATH = '/Users/jaytlinaskew/GitRepository/TimeSeries-Analysis/data/processed/MitreDictionary.json'

# Precompiled stop words for preprocessing
STOP_WORDS = set(stopwords.words('english'))

# Load MITRE ATT&CK techniques and build searchable text
with open(MITRE_DICT_PATH, 'r') as file:
    mitre_data = json.load(file)

technique_id_to_text = {
    tech_id: f"{details.get('name', '')} {details.get('description', '')} {' '.join(details.get('keywords', []))}".lower()
    for tech_id, details in mitre_data.items()
}

def preprocess_text(text):

    words = text.lower().split()
    return ' '.join(word for word in words if word not in STOP_WORDS)

def find_matching_techniques(input_text, threshold=75, top_n=5):

    processed_input = preprocess_text(input_text)
    search_space = list(technique_id_to_text.values())

    matches = process.extract(processed_input, search_space, limit=top_n)

    # Reverse lookup to find the corresponding technique ID
    text_to_id = {v: k for k, v in technique_id_to_text.items()}
    final_matches = [
        (text_to_id[full_text], score)
        for full_text, score in matches
        if score >= threshold
    ]

    return final_matches

if __name__ == "__main__":
    input_text = (
        "abuse msiexec.exe to proxy execution of malicious payloads. Msiexec.exe is the command-line utility for the Windows Installer and is thus commonly associated with executing installation packages (.msi). The Msiexec.exe binary also be digitally signed by Microsoft. abuse msiexec.exe to launch local or network accessible MSI files. Msiexec.exe also execute DLLs. Since it be signed and native on Windows systems, msiexec.exe be used to bypass application control solutions that do not account for its potential abuse. Msiexec.exe execution also be elevated to SYSTEM privileges if the AlwaysInstallElevated policy is enabled"
    )

    matches = find_matching_techniques(input_text)

    print("\nMatching MITRE ATT&CK Techniques:")
    for match_id, score in matches:
        print(f"- ID: {match_id} (Score: {score})")



Matching MITRE ATT&CK Techniques:
- ID: T1218.007 (Score: 95)
- ID: T1055.011 (Score: 86)
- ID: T1053.005 (Score: 86)
- ID: T1205.002 (Score: 86)
- ID: T1560.001 (Score: 86)
